<a href="https://colab.research.google.com/github/williamscarson093-cloud/steam-smart-buy/blob/main/steam_smart_buy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Steam Smart Buy

My app helps money conscious gamers decide which Steam game to buy by comparing games that fit their budget and genre, checking current deals, and weighing review.

In [ ]:
from google.colab import drive
import pandas as pd
import os

drive.mount('/content/drive')

# Find the Steam games dataset
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    if 'steam_games.csv' in files:
        path = os.path.join(root, 'steam_games.csv')
        break
else:
    raise FileNotFoundError("steam_games.csv not found in Drive")

df = pd.read_csv(path)
print(f"Loaded {len(df)} games")

Mounted at /content/drive
Loaded 1000 games


In [ ]:
print(df.columns.tolist())

['appid', 'name', 'release_date', 'release_year', 'price', 'discount', 'required_age', 'windows', 'mac', 'linux', 'developers', 'publishers', 'genres', 'n_languages', 'achievements', 'metacritic_score', 'positive', 'negative', 'pct_pos_total', 'num_reviews_total', 'estimated_owners', 'owners_low', 'avg_playtime_hours', 'peak_ccu', 'short_description', 'critic_reviews']


In [ ]:
def popular_games(games):
    popular = games[games["num_reviews_total"] >= 10000]
    return popular[["name", "num_reviews_total"]]

print(popular_games(df).head(5))

                              name  num_reviews_total
0                 Counter-Strike 2            8632939
1              PUBG: BATTLEGROUNDS            2513842
2                           Dota 2            2452595
3        Grand Theft Auto V Legacy            1803832
4  Tom Clancy's Rainbow Six® Siege            1168020


In [ ]:
def find_games(games, genre, budget):
    matches = games[
        (games["genres"].str.contains(genre, case=False, na=False, regex=False)) &
        (games["price"] <= budget)
    ]

    matches = matches.sort_values("pct_pos_total", ascending=False)

    return matches[["name", "genres", "price", "pct_pos_total"]]

results = find_games(df, "Action", 20)
print(results.head(10))

                          name  \
473  HoloCure - Save the Fans!   
63           Vampire Survivors   
311                Katana ZERO   
320                Pizza Tower   
54           People Playground   
6                     Terraria   
878            The Expendabros   
711                Wobbly Life   
605          The Wolf Among Us   
125     Mount & Blade: Warband   

                                                genres  price  pct_pos_total  
473                   Action, Adventure, Casual, Indie   0.00             99  
63                          Action, Casual, Indie, RPG   4.99             98  
311                                      Action, Indie  14.99             98  
320                                      Action, Indie  19.99             98  
54                   Action, Casual, Indie, Simulation   9.99             98  
6                        Action, Adventure, Indie, RPG   9.99             97  
878             Action, Adventure, Indie, Free To Play   0.00           

In [ ]:

import requests

def check_deals(game_name):
    url = "https://www.cheapshark.com/api/1.0/games"

    params = {"title": game_name}

    headers = {
        "User-Agent": "SteamSmartBuy/1.0 (student project)"
    }

    response = requests.get(url, params=params, headers=headers, timeout=15)

    print("Status:", response.status_code)

    if response.status_code != 200:
        print("API error:", response.text[:300])
        return []

    deals = response.json()

    for game in deals[:5]:
        print("Game:", game["external"])
        print("Price: $", game["cheapest"])
        print("-------------------")

    return deals

deals = check_deals("Terraria")


Status: 200
Game: Terraria
Price: $ 9.99
-------------------


In [ ]:

def recommend_games(games, genre, budget):
    matches = find_games(games, genre, budget)

    if matches.empty:
        print("No games found.")
        return matches

    print("Top games for your budget:")
    print(matches.head(5))

    return matches.head(5)

recommendations = recommend_games(df, "Action", 20)


Top games for your budget:
                          name                             genres  price  \
473  HoloCure - Save the Fans!   Action, Adventure, Casual, Indie   0.00   
63           Vampire Survivors         Action, Casual, Indie, RPG   4.99   
311                Katana ZERO                      Action, Indie  14.99   
320                Pizza Tower                      Action, Indie  19.99   
54           People Playground  Action, Casual, Indie, Simulation   9.99   

     pct_pos_total  
473             99  
63              98  
311             98  
320             98  
54              98  


In [ ]:

class SteamSmartBuy:
    def __init__(self, games):
        self.games = games

    def recommend(self, genre, budget):
        return recommend_games(self.games, genre, budget)

    def deals(self, game_name):
        return check_deals(game_name)


app = SteamSmartBuy(df)

app.recommend("Action", 20)
app.deals("Terraria")


Top games for your budget:
                          name                             genres  price  \
473  HoloCure - Save the Fans!   Action, Adventure, Casual, Indie   0.00   
63           Vampire Survivors         Action, Casual, Indie, RPG   4.99   
311                Katana ZERO                      Action, Indie  14.99   
320                Pizza Tower                      Action, Indie  19.99   
54           People Playground  Action, Casual, Indie, Simulation   9.99   

     pct_pos_total  
473             99  
63              98  
311             98  
320             98  
54              98  
Status: 200
Game: Terraria
Price: $ 9.99
-------------------


[{'gameID': '39',
  'steamAppID': '105600',
  'cheapest': '9.99',
  'cheapestDealID': '%2B7y%2FjZTRXxyTajQx%2FtvBN1%2BoisI6Iv5D7TL9ma6o7lU%3D',
  'external': 'Terraria',
  'internalName': 'TERRARIA',
  'thumb': 'https://shared.fastly.steamstatic.com/store_item_assets/steam/apps/105600/capsule_231x87.jpg?t=1792530636'}]